# Python for Bioinformatics — Part II Solutions
## Modules 6–8

Reference solutions with commentary. Run from the folder containing `data/`. More than one correct approach usually exists.

First, the two parsers we reuse throughout (from the Module 6 worked examples).

In [ ]:
def read_fasta(path):
    sequences, header, parts = {}, None, []
    with open(path) as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if header is not None:
                    sequences[header] = "".join(parts)
                header = line[1:]; parts = []
            else:
                parts.append(line)
        if header is not None:
            sequences[header] = "".join(parts)
    return sequences

def read_fastq(path):
    reads = []
    with open(path) as f:
        lines = [ln.rstrip("\n") for ln in f]
    for i in range(0, len(lines), 4):
        reads.append((lines[i][1:], lines[i+1], lines[i+3]))
    return reads
print("parsers ready")

## Module 6

**6A.1**

In [ ]:
total = headers = 0
with open("data/sample.fasta") as f:
    for line in f:
        line = line.rstrip("\n")
        total += 1
        if line.startswith(">"):
            headers += 1
print(f"{total} lines, {headers} headers")

**6A.2**

In [ ]:
with open("data/sample.fasta") as f:
    lines = f.readlines()
print("lines:", len(lines))
for line in lines:
    line = line.rstrip("\n")
    if line.startswith(">"):
        print(line[1:])

**6B.1**

In [ ]:
records = read_fasta("data/sample.fasta")
print("sequences:", len(records))
for header, seq in records.items():
    if header.startswith("katG"):
        print("katG length:", len(seq))

**6B.2**

In [ ]:
records = read_fasta("data/sample.fasta")
for header, seq in records.items():
    gene = header.split()[0]
    gc = (seq.count("G") + seq.count("C")) / len(seq)
    print(f"{gene}: {gc:.1%} GC")

**6C.1**

In [ ]:
def phred(char):
    return ord(char) - 33
print(phred("I"), phred("#"))   # 40 2

**6C.2**

In [ ]:
reads = read_fastq("data/sample.fastq")
for header, seq, qual in reads:
    mean_q = sum(ord(c) - 33 for c in qual) / len(qual)
    print(f"{header}: {mean_q:.1f}")

**6C.3** — quality filter (read3 drops out).

In [ ]:
reads = read_fastq("data/sample.fastq")
for header, seq, qual in reads:
    mean_q = sum(ord(c) - 33 for c in qual) / len(qual)
    if mean_q >= 30:
        print(f"{header}: {mean_q:.1f}")

**6D.1**

In [ ]:
import csv
with open("data/variants.csv") as f:
    for row in csv.DictReader(f):
        print(row["gene"], row["alt"])

**6D.2** — convert string to float, then filter.

In [ ]:
import csv
with open("data/variants.csv") as f:
    for row in csv.DictReader(f):
        q = float(row["quality"])
        if q >= 40.0:
            print(row["gene"], row["position"], q)

**Module 6 challenge** — FASTA → GC report writer, then read back.

In [ ]:
records = read_fasta("data/sample.fasta")
with open("gc_report.tsv", "w") as f:
    f.write("gene\tgc\n")
    for header, seq in records.items():
        gene = header.split()[0]
        gc = (seq.count("G") + seq.count("C")) / len(seq)
        f.write(f"{gene}\t{gc:.3f}\n")

print("--- gc_report.tsv ---")
with open("gc_report.tsv") as f:
    print(f.read())

## Module 7

**7A.1** — deliberate errors, wrapped so the notebook keeps running.

In [ ]:
d = {"A": 1}
try:
    d["Z"]
except KeyError as e:
    print("KeyError:", e)
try:
    int("abc")
except ValueError as e:
    print("ValueError:", e)
try:
    1 / 0
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e)

**7B.1**

In [ ]:
def safe_open(path):
    try:
        with open(path) as f:
            return len(f.read())
    except FileNotFoundError:
        return "missing"

print(safe_open("data/sample.fasta"))
print(safe_open("data/nope.txt"))

**7B.2** — validate and raise.

In [ ]:
def gc_content(seq):
    seq = seq.upper()
    if len(seq) == 0:
        raise ValueError("Empty sequence has no GC content")
    if not set(seq) <= {"A", "C", "G", "T"}:
        bad = set(seq) - {"A", "C", "G", "T"}
        raise ValueError(f"Invalid bases: {bad}")
    return (seq.count("G") + seq.count("C")) / len(seq)

print(gc_content("ATGC"))
for test in ["ATBX", ""]:
    try:
        gc_content(test)
    except ValueError as e:
        print(f"{test!r} ->", e)

**7C.1** — the fix: only count complete codons.

In [ ]:
def count_codons(seq):
    count = 0
    for i in range(0, len(seq), 3):
        if len(seq[i:i+3]) == 3:    # only complete codons
            count += 1
    return count
# cleaner equivalent: len(seq) // 3
print(count_codons("ATGAAAC"))     # 2
print("alt:", len("ATGAAAC") // 3) # 2

**7C.2** — assertion before translating.

In [ ]:
def translate_frame(seq):
    assert len(seq) % 3 == 0, f"length {len(seq)} not a multiple of 3"
    return [seq[i:i+3] for i in range(0, len(seq), 3)]

print(translate_frame("ATGAAA"))
try:
    translate_frame("ATGAA")
except AssertionError as e:
    print("AssertionError:", e)

**7D.1** — assert checks.

In [ ]:
def gc_content(seq):
    seq = seq.upper()
    return (seq.count("G") + seq.count("C")) / len(seq)

assert gc_content("GCGC") == 1.0
assert gc_content("ATAT") == 0.0
assert gc_content("ATGC") == 0.5
assert gc_content("atgc") == 0.5     # lowercase handled
print("all checks passed")

**7D.2** — pytest-style functions, called manually.

In [ ]:
def gc_content(seq):
    seq = seq.upper()
    if not seq:
        raise ValueError("empty")
    if not set(seq) <= set("ACGT"):
        raise ValueError("invalid")
    return (seq.count("G") + seq.count("C")) / len(seq)

def test_gc_basic():
    assert gc_content("GCGC") == 1.0
    assert gc_content("ATAT") == 0.0

def test_gc_lowercase():
    assert gc_content("gc") == 1.0

def test_gc_invalid_raises():
    try:
        gc_content("ATBX")
    except ValueError:
        return
    raise AssertionError("expected ValueError")

for test in [test_gc_basic, test_gc_lowercase, test_gc_invalid_raises]:
    test()
    print(test.__name__, "PASSED")

**Module 7 challenge** — hardened seqtools + tests.

In [ ]:
def _validate(seq):
    seq = seq.upper()
    if not seq:
        raise ValueError("empty sequence")
    if not set(seq) <= set("ACGT"):
        raise ValueError(f"invalid bases: {set(seq) - set('ACGT')}")
    return seq

def gc_content(seq):
    seq = _validate(seq)
    return (seq.count("G") + seq.count("C")) / len(seq)

def reverse_complement(seq):
    seq = _validate(seq)
    return seq.translate(str.maketrans("ACGT", "TGCA"))[::-1]

def test_gc():
    assert gc_content("GCGC") == 1.0
    assert gc_content("ATGC") == 0.5

def test_revcomp():
    assert reverse_complement("ATGC") == "GCAT"
    assert reverse_complement("ATGCAT") == "ATGCAT"   # palindrome

def test_errors():
    for fn in (gc_content, reverse_complement):
        try:
            fn("ATBX")
        except ValueError:
            continue
        raise AssertionError(f"{fn.__name__} should reject invalid input")

passed = 0
for test in [test_gc, test_revcomp, test_errors]:
    test(); passed += 1
print(f"{passed}/3 tests passed")

## Module 8

**8A.1** — Sequence with reverse_complement returning a new Sequence.

In [ ]:
class Sequence:
    def __init__(self, identifier, seq):
        self.id = identifier
        self.seq = seq.upper()
    def __len__(self):
        return len(self.seq)
    def gc_content(self):
        if not self.seq: return None
        return (self.seq.count("G") + self.seq.count("C")) / len(self.seq)
    def reverse_complement(self):
        rc = self.seq.translate(str.maketrans("ACGT", "TGCA"))[::-1]
        return Sequence(self.id + "_rc", rc)

print(Sequence("g1", "ATGC").reverse_complement().seq)   # GCAT

**8A.2** — transcribe and composition.

In [ ]:
class Sequence:
    def __init__(self, identifier, seq):
        self.id, self.seq = identifier, seq.upper()
    def transcribe(self):
        return self.seq.replace("T", "U")
    def composition(self):
        counts = {}
        for base in self.seq:
            counts[base] = counts.get(base, 0) + 1
        return counts

s = Sequence("g1", "ATGCGT")
print(s.transcribe())       # AUGCGU
print(s.composition())

**8B.1**

In [ ]:
class Sequence:
    def __init__(self, identifier, seq):
        self.id, self.seq = identifier, seq.upper()
    def __len__(self):
        return len(self.seq)
    def __str__(self):
        return f"{self.id}: {self.seq}"

print(Sequence("g1", "atgc"))   # g1: ATGC

**8B.2** — equality by sequence.

In [ ]:
class Sequence:
    def __init__(self, identifier, seq):
        self.id, self.seq = identifier, seq.upper()
    def __eq__(self, other):
        return self.seq == other.seq

print(Sequence("a", "ATGC") == Sequence("b", "atgc"))   # True
print(Sequence("a", "ATGC") == Sequence("c", "TTTT"))   # False

**8C.1**

In [ ]:
class Sequence:
    def __init__(self, identifier, seq):
        self.id, self.seq = identifier, seq.upper()
    def gc_content(self):
        return (self.seq.count("G") + self.seq.count("C")) / len(self.seq) if self.seq else None

class FastqRead(Sequence):
    def __init__(self, identifier, seq, qual):
        super().__init__(identifier, seq)
        self.qual = qual
    def mean_quality(self):
        return sum(ord(c) - 33 for c in self.qual) / len(self.qual)

header, seq, qual = read_fastq("data/sample.fastq")[0]
r = FastqRead(header, seq, qual)
print(r.id, r.gc_content(), r.mean_quality())

**8C.2** — passes_filter.

In [ ]:
class FastqRead(Sequence):
    def __init__(self, identifier, seq, qual):
        super().__init__(identifier, seq)
        self.qual = qual
    def mean_quality(self):
        return sum(ord(c) - 33 for c in self.qual) / len(self.qual)
    def passes_filter(self, min_q=30):
        return self.mean_quality() >= min_q

reads = [FastqRead(h, s_, q) for h, s_, q in read_fastq("data/sample.fastq")]
for r in reads:
    print(r.id, round(r.mean_quality(), 1), r.passes_filter())

**Part II capstone** — object-oriented, validated, tested FASTA toolkit.

In [ ]:
class Sequence:
    def __init__(self, identifier, seq):
        seq = seq.upper()
        if not set(seq) <= set("ACGT"):
            raise ValueError(f"{identifier}: invalid bases {set(seq) - set('ACGT')}")
        self.id = identifier
        self.seq = seq
    def __len__(self):
        return len(self.seq)
    def __str__(self):
        return f"{self.id} ({len(self)} bp)"
    def gc_content(self):
        return (self.seq.count("G") + self.seq.count("C")) / len(self.seq) if self.seq else None
    def reverse_complement(self):
        rc = self.seq.translate(str.maketrans("ACGT", "TGCA"))[::-1]
        return Sequence(self.id + "_rc", rc)

def load_fasta(path):
    seqs = []
    for header, seq in read_fasta(path).items():
        seqs.append(Sequence(header.split()[0], seq))
    return seqs

def test_gc():
    assert Sequence("t", "GCGC").gc_content() == 1.0

def test_revcomp():
    assert Sequence("t", "ATGC").reverse_complement().seq == "GCAT"

def test_invalid_raises():
    try:
        Sequence("t", "ATBX")
    except ValueError:
        return
    raise AssertionError("expected ValueError")

for test in [test_gc, test_revcomp, test_invalid_raises]:
    test()
print("3/3 tests passed\n")

print("--- FASTA report ---")
for s_ in load_fasta("data/sample.fasta"):
    print(f"{s_.id:6s} {len(s_):4d} bp   GC {s_.gc_content():.1%}")

---
*Part II complete.* You can read real files, write robust validated code with tests, and structure it with classes — the skeleton of a real tool. Part III brings the ecosystem (NumPy, pandas, matplotlib, Biopython) that makes it fast and professional.